In [1]:
"""
Pure NumPy implementation of an LSTM, matching the gate math:

    i_t = sigmoid(W_i @ x_t + U_i @ h_{t-1} + b_i)   # input gate
    f_t = sigmoid(W_f @ x_t + U_f @ h_{t-1} + b_f)   # forget gate
    g_t = tanh   (W_g @ x_t + U_g @ h_{t-1} + b_g)   # candidate cell content
    o_t = sigmoid(W_o @ x_t + U_o @ h_{t-1} + b_o)   # output gate

    c_t = f_t * c_{t-1} + i_t * g_t                  # elementwise
    h_t = o_t * tanh(c_t)                            # elementwise

No autograd, no training loop -- forward pass only, meant to make every
matrix multiply and elementwise op explicit and inspectable.
"""

import numpy as np


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


class LSTMCell:
    """One LSTM cell: input_dim -> hidden_dim, run one timestep at a time."""

    def __init__(self, input_dim, hidden_dim, seed=None):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        rng = np.random.default_rng(seed)

        # W_* : (hidden_dim, input_dim)   -- maps x_t
        # U_* : (hidden_dim, hidden_dim)  -- maps h_{t-1}
        # b_* : (hidden_dim,)
        scale = 1.0 / np.sqrt(hidden_dim)
        shapes_W = (hidden_dim, input_dim)
        shapes_U = (hidden_dim, hidden_dim)

        self.W_i = rng.uniform(-scale, scale, shapes_W)
        self.W_f = rng.uniform(-scale, scale, shapes_W)
        self.W_g = rng.uniform(-scale, scale, shapes_W)
        self.W_o = rng.uniform(-scale, scale, shapes_W)

        self.U_i = rng.uniform(-scale, scale, shapes_U)
        self.U_f = rng.uniform(-scale, scale, shapes_U)
        self.U_g = rng.uniform(-scale, scale, shapes_U)
        self.U_o = rng.uniform(-scale, scale, shapes_U)

        self.b_i = np.zeros(hidden_dim)
        # Keras-style trick: bias the forget gate toward "remember" at init
        self.b_f = np.ones(hidden_dim)
        self.b_g = np.zeros(hidden_dim)
        self.b_o = np.zeros(hidden_dim)

    def set_weights(self, **kwargs):
        """Manually override any weight/bias, e.g. cell.set_weights(W_i=..., b_f=...)."""
        for name, value in kwargs.items():
            setattr(self, name, np.asarray(value, dtype=float))

    def step(self, x_t, h_prev, c_prev):
        """
        One timestep. x_t: (input_dim,), h_prev/c_prev: (hidden_dim,).
        Returns h_t, c_t, and a dict of every intermediate value for inspection.
        """
        i_t = sigmoid(self.W_i @ x_t + self.U_i @ h_prev + self.b_i)
        f_t = sigmoid(self.W_f @ x_t + self.U_f @ h_prev + self.b_f)
        g_t = np.tanh(self.W_g @ x_t + self.U_g @ h_prev + self.b_g)
        o_t = sigmoid(self.W_o @ x_t + self.U_o @ h_prev + self.b_o)

        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * np.tanh(c_t)

        cache = dict(i_t=i_t, f_t=f_t, g_t=g_t, o_t=o_t, c_t=c_t, h_t=h_t)
        return h_t, c_t, cache

    def forward_sequence(self, X, h0=None, c0=None):
        """
        Run the cell over a full sequence.
        X: (seq_len, input_dim)
        Returns:
            H: (seq_len, hidden_dim)  -- hidden state at every timestep
            C: (seq_len, hidden_dim)  -- cell state at every timestep
            caches: list of per-timestep intermediate dicts
        """
        seq_len = X.shape[0]
        h = np.zeros(self.hidden_dim) if h0 is None else h0
        c = np.zeros(self.hidden_dim) if c0 is None else c0

        H = np.zeros((seq_len, self.hidden_dim))
        C = np.zeros((seq_len, self.hidden_dim))
        caches = []

        for t in range(seq_len):
            h, c, cache = self.step(X[t], h, c)
            H[t] = h
            C[t] = c
            caches.append(cache)

        return H, C, caches


class BidirectionalLSTM:
    """
    Mirrors Bidirectional(LSTM(units)) with default merge_mode='concat':
    runs one LSTMCell forward over the sequence and an independent LSTMCell
    backward over the reversed sequence, then concatenates the two final
    hidden states.
    """

    def __init__(self, input_dim, hidden_dim, seed=None):
        self.fwd_cell = LSTMCell(input_dim, hidden_dim, seed=seed)
        self.bwd_cell = LSTMCell(input_dim, hidden_dim, seed=None if seed is None else seed + 1)
        self.hidden_dim = hidden_dim

    def forward_sequence(self, X):
        """
        X: (seq_len, input_dim)
        Returns:
            h_concat: (2 * hidden_dim,)  -- final output, forward_h_T ++ backward_h_T
            H_fwd, H_bwd: (seq_len, hidden_dim) each, full per-timestep hidden states
        """
        H_fwd, _, _ = self.fwd_cell.forward_sequence(X)
        H_bwd, _, _ = self.bwd_cell.forward_sequence(X[::-1])
        H_bwd = H_bwd[::-1]  # re-align to original time order

        h_concat = np.concatenate([H_fwd[-1], H_bwd[-1]])
        return h_concat, H_fwd, H_bwd


def dense_sigmoid(x, W, b):
    """Final Dense(1, activation='sigmoid') layer."""
    z = W @ x + b
    return sigmoid(z)


# ---------------------------------------------------------------------------
# Self-test: reproduce the toy example worked through by hand earlier
# (3-dim input, 2 hidden units, single forward step from a known state).
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    cell = LSTMCell(input_dim=3, hidden_dim=2)

    cell.set_weights(
        W_i=[[0.1, 0.2, 0.0], [0.0, 0.1, -0.1]],
        W_f=[[0.2, -0.1, 0.1], [0.1, 0.0, 0.2]],
        W_g=[[0.3, 0.1, -0.2], [-0.1, 0.2, 0.1]],
        W_o=[[0.1, 0.1, 0.1], [0.2, -0.1, 0.0]],
        U_i=[[0.1, -0.2], [0.05, 0.1]],
        U_f=[[0.2, 0.1], [-0.1, 0.05]],
        U_g=[[-0.1, 0.1], [0.2, 0.0]],
        U_o=[[0.1, 0.0], [0.0, 0.1]],
        b_i=[0.0, 0.0],
        b_f=[0.0, 0.0],  # override the default 1.0 init to match the hand-worked example
        b_g=[0.0, 0.0],
        b_o=[0.0, 0.0],
    )

    x_t = np.array([1.0, 0.0, -1.0])
    h_prev = np.array([0.1, -0.2])
    c_prev = np.array([0.3, 0.05])

    h_t, c_t, cache = cell.step(x_t, h_prev, c_prev)

    print("i_t =", np.round(cache["i_t"], 3), " expected [0.537, 0.521]")
    print("f_t =", np.round(cache["f_t"], 3), " expected [0.525, 0.470]")
    print("g_t =", np.round(cache["g_t"], 3), " expected [0.438, -0.178]")
    print("o_t =", np.round(cache["o_t"], 3), " expected [0.502, 0.545]")
    print("c_t =", np.round(c_t, 3), "        expected [0.393, -0.069]")
    print("h_t =", np.round(h_t, 3), "        expected [0.188, -0.038]")

    # --- Full pipeline sanity check on a short random sequence ---
    print("\nFull sequence + Bidirectional + Dense(1) demo:")
    seq_len, embed_dim, hidden_units = 5, 8, 16
    rng = np.random.default_rng(0)
    X = rng.normal(size=(seq_len, embed_dim))  # stand-in for an embedded sentence

    bilstm = BidirectionalLSTM(input_dim=embed_dim, hidden_dim=hidden_units, seed=42)
    h_final, H_fwd, H_bwd = bilstm.forward_sequence(X)
    print("concatenated final hidden state shape:", h_final.shape)  # (32,)

    W_dense = rng.normal(scale=0.1, size=(1, 2 * hidden_units))
    b_dense = np.zeros(1)
    y_hat = dense_sigmoid(h_final, W_dense, b_dense)
    print("sentiment prediction (sigmoid output):", y_hat)

i_t = [0.537 0.521]  expected [0.537, 0.521]
f_t = [0.525 0.47 ]  expected [0.525, 0.470]
g_t = [ 0.438 -0.178]  expected [0.438, -0.178]
o_t = [0.502 0.545]  expected [0.502, 0.545]
c_t = [ 0.393 -0.069]         expected [0.393, -0.069]
h_t = [ 0.188 -0.038]         expected [0.188, -0.038]

Full sequence + Bidirectional + Dense(1) demo:
concatenated final hidden state shape: (32,)
sentiment prediction (sigmoid output): [0.50003521]
